In [227]:
import pandas as pd

In [228]:
df = pd.read_csv('expenses.csv', thousands=',')
people = ['ben','sherina','keiton','chris','hyeok','mandy','eric', 'harry','haeyung']

df = df.melt(id_vars=[col for col in df.columns if col not in people], var_name='owed_by', value_name='portion')

# Check data types of relevant columns
print(df[['amount', 'portion', 'TOTAL']].dtypes)
df['amount'] = df['amount'].astype(float)

# Ensure numeric types for calculations
df['amount'] = pd.to_numeric(df['amount'], errors='ignore')
df['portion'] = pd.to_numeric(df['portion'], errors='ignore')
df['TOTAL'] = pd.to_numeric(df['TOTAL'], errors='ignore')

#lower case and trim whitespace
df['paid_by'] = df['paid_by'].str.lower().str.strip()
df['owed_by'] = df['owed_by'].str.lower().str.strip()


df = df[(df['portion']!=0) & (df['portion'].notna())]

amount     float64
portion    float64
TOTAL      float64
dtype: object


In [229]:
df['owed_amount'] = df['amount'] * df['portion'] / df['TOTAL']


In [230]:
for person in people:
    print(f"{person} owes: {df[df['owed_by']==person]['owed_amount'].sum():.2f}")
    print(f"{person} paid: {df[df['paid_by']==person]['owed_amount'].sum():.2f}")


ben owes: 1152.78
ben paid: 5250.78
sherina owes: 1151.69
sherina paid: 463.29
keiton owes: 1131.96
keiton paid: 582.54
chris owes: 1191.45
chris paid: 537.57
hyeok owes: 1352.39
hyeok paid: 66.33
mandy owes: 280.02
mandy paid: 0.00
eric owes: 346.96
eric paid: 364.53
harry owes: 313.36
harry paid: 0.00
haeyung owes: 344.42
haeyung paid: 0.00


In [231]:
df = df[df['owed_amount']!=0]
# df = df[df['paid_by']!=df['owed_by']]
df.reset_index(drop = True, inplace=True)
df.head(3)

,Item,description,amount,paid_by,TOTAL,owed_by,portion,owed_amount
0,BNB,by /person/nights,2758.52,ben,1.0,ben,0.164948,455.013609
1,Suburu + fuel,by /person/day,582.54,keiton,1.0,ben,0.160377,93.426226
2,Van + fuel,by /person/day,1383.20,ben,1.0,ben,0.160377,221.833962


In [232]:
for person in people:
    print(f"{person} owes: {df[df['owed_by']==person]['owed_amount'].sum():.2f}")
    print(f"{person} paid: {df[df['paid_by']==person]['owed_amount'].sum():.2f}")


ben owes: 1152.78
ben paid: 5250.78
sherina owes: 1151.69
sherina paid: 463.29
keiton owes: 1131.96
keiton paid: 582.54
chris owes: 1191.45
chris paid: 537.57
hyeok owes: 1352.39
hyeok paid: 66.33
mandy owes: 280.02
mandy paid: 0.00
eric owes: 346.96
eric paid: 364.53
harry owes: 313.36
harry paid: 0.00
haeyung owes: 344.42
haeyung paid: 0.00


In [233]:
from collections import defaultdict
# Step 1: Calculate net balances for each person
balances = defaultdict(float)
for _, row in df.iterrows():
    balances[row['paid_by']] += row['owed_amount']
    balances[row['owed_by']] -= row['owed_amount']
balances

defaultdict(float,
            {'ben': 4098.003407312358,
             'keiton': -549.4189707422585,
             'chris': -653.8823022641816,
             'hyeok': -1286.061346192613,
             'sherina': -688.4045670173422,
             'eric': 17.566315090182577,
             'haeyung': -344.419473952018,
             'harry': -313.36484536998466,
             'mandy': -280.01821686414644})

In [234]:
sugar_daddy_lookup = {'sherina': 'ben', 'haeyung': 'keiton', 'mandy':'eric'}
for person, balance in balances.items():
    if person in sugar_daddy_lookup:
        sugar_baby = person
        sugar_daddy = sugar_daddy_lookup[sugar_baby]
        balances[sugar_daddy] += balances[sugar_baby]
        balances[sugar_baby] = 0
balances

defaultdict(float,
            {'ben': 3409.5988402950156,
             'keiton': -893.8384446942764,
             'chris': -653.8823022641816,
             'hyeok': -1286.061346192613,
             'sherina': 0,
             'eric': -262.45190177396387,
             'haeyung': 0,
             'harry': -313.36484536998466,
             'mandy': 0})

In [235]:

 # Step 2: Separate into creditors and debtors
creditors = []
debtors = []
for person, balance in balances.items():
    if balance > 0:
        creditors.append((person, balance))
    elif balance < 0:
        debtors.append((person, -balance))

In [236]:
# Step 3: Minimize transactions
minimized_transactions = []
while creditors and debtors:
    creditor, credit_amount = creditors.pop()
    debtor, debt_amount = debtors.pop()

    payment = min(credit_amount, debt_amount)
    minimized_transactions.append({'paid_by': creditor, 'owed_by': debtor, 'owed_amount': payment})

    if credit_amount > payment:
        creditors.append((creditor, credit_amount - payment))
    if debt_amount > payment:
        debtors.append((debtor, debt_amount - payment))


In [237]:
for transaction in minimized_transactions:
    print(transaction['owed_by'], 'pays', round(transaction['owed_amount'],2), 'to', transaction['paid_by'])

harry pays 313.36 to ben
eric pays 262.45 to ben
hyeok pays 1286.06 to ben
chris pays 653.88 to ben
keiton pays 893.84 to ben


In [238]:

# Generate report to README.md
import datetime

report = "# Expense Splitting Report\n\n"
report += "Google Sheets Link\n"
report += "https://docs.google.com/spreadsheets/d/1sgjZCzSm74SpFO3mT2y9Xk_OrHESxQ3xAgAuiUvRKkQ/edit?gid=1818656043#gid=1818656043\n\n"
report += "owed_amount = amount * portion / total\n\n"
report += f"Report generated on {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}\n\n"


# Total expenses
total_expenses = df['owed_amount'].sum()
report += f"## Total Expenses\n{total_expenses:.2f}\n\n"

# Amount paid and owed
paid_by_person = df.groupby('paid_by')['owed_amount'].sum()
owed_by_person = df.groupby('owed_by')['owed_amount'].sum()

report += "## Summary by Person\n\n"
report += "| Person | Paid | Owes | Net Balance |\n"
report += "|--------|------|------|-------------|\n"
for person in people:
    paid = paid_by_person.get(person, 0)
    owed = owed_by_person.get(person, 0)
    net = paid-owed
    balance_str = f"{net:.2f}" if net != 0 else "0.00"
    report += f"| {person} | {paid:.2f} | {owed:.2f} | {balance_str} |\n"
report += "\n"

report += "## Minimized Transactions\n"


for transaction in minimized_transactions:
    report += f"- {transaction['owed_by']} pays {round(transaction['owed_amount'],2)} to {transaction['paid_by']}\n"
    
report += "#### Note\n"
for sugar_baby, sugar_daddy in sugar_daddy_lookup.items():
    report += f"- Sugar daddy {sugar_daddy} pays for sugar baby {sugar_baby}\n"
    
report += '## Venmo: @benzhong\n'
with open('README.md', 'w') as f:
    f.write(report)
print("Report generated and saved to README.md")

Report generated and saved to README.md
